## Background

Surrogate optimization — replacing expensive simulations with trained neural networks inside
an optimization model — is standard in process systems engineering (Misener & Biegler, 2023).
The dominant approach uses **ReLU activations encoded as MILP**: one binary variable per neuron,
solved by branch-and-bound (Grimstad & Andersson, 2019; OMLT, Ceccon et al., 2022).

This notebook explores a different path: **smooth activations (Softplus)** embedded as Gurobi
nonlinear constraints, optimized with the **nonlinear barrier solver**. The hypothesis is that
smooth differentiable surrogates are more naturally compatible with continuous NLP methods than
piecewise-linear MILP embeddings, particularly for processes governed by smooth, continuous physics.

> *Can smooth neural surrogates be optimised efficiently with Gurobi nonlinear barrier methods?*

The workflow is:
$$\underbrace{\text{Engineering simulator}}_{\text{NatLabRockies electrolyzer}} \;\longrightarrow\; \text{Training data} \;\longrightarrow\; \underbrace{\text{Smooth surrogate}}_{\text{Softplus NN}} \;\longrightarrow\; \underbrace{\text{Optimise}}_{\text{Gurobi NLP barrier}}$$

This is a simplified educational demonstration, **not** a reproduction of any specific paper.
It is inspired by surrogate optimization for hydrogen and Power-to-X systems (Vo et al., 2025;
Ghilardi et al., 2025), but scoped to a single-stack **operational dispatch** problem.
Electrolyzer physics (polarisation curves, Nernst voltage, ohmic losses) is smooth and continuous
 smooth neural surrogates are a natural modelling choice. Dispatch problems are also
typically solved repeatedly over rolling horizons, making fast local NLP solves attractive.

**Note on local optimality:** The nonlinear barrier solver finds *locally* optimal solutions.
This is appropriate for operational scheduling where good feasible solutions and short solve
times matter more than global optimality guarantees.

**References:**
Ceccon et al. (2022) [OMLT](https://arxiv.org/abs/2202.02414) ·
Grimstad & Andersson (2019) [arXiv](https://arxiv.org/abs/1907.03140) ·
Misener & Biegler (2023) [doi](https://doi.org/10.1016/j.compchemeng.2023.108411) ·
Tsay (2021) [doi](https://doi.org/10.1016/j.compchemeng.2021.107452) ·
Koksal & Aydin (2023) [arXiv](https://arxiv.org/abs/2302.00990) ·
Vo et al. (2025) [doi](https://doi.org/10.1016/j.apenergy.2025.125969) ·
[Gurobi ML](https://gurobi-machinelearning.readthedocs.io/en/stable/) ·
[Gurobi nonlinear](https://docs.gurobi.com/projects/optimizer/en/current/features/nonlinear.html) ·
[NatLabRockies electrolyzer](https://github.com/NatLabRockies/electrolyzer)

In [ ]:
!uv pip install "git+https://github.com/NatLabRockies/electrolyzer.git"

In [ ]:
import os
import tempfile
import time
import matplotlib.pyplot as plt
import numpy as np
import torch
import torch.nn as nn
from torch.utils.data import DataLoader, TensorDataset
import gurobipy as gp
from gurobipy import GRB
from gurobi_ml import add_predictor_constr
from electrolyzer.simulation.bert import run_electrolyzer

## Step 1 — Data from the Electrolyzer Simulator

Data is generated from the [NatLabRockies electrolyzer](https://github.com/NatLabRockies/electrolyzer)
package, which implements PEM and alkaline cell models with Nernst equation, Butler-Volmer
activation overpotential, and ohmic losses. A smooth neural surrogate is trained to approximate
the simulator and embedded into Gurobi for dispatch optimisation.

The surrogate maps two inputs — stack power $P$ (kW) and stack temperature $T$ (°C) — to
hydrogen production rate $\dot{m}_{\text{H}_2}$ (kg/h). Using two inputs creates a 2-D surface
that motivates the surrogate (and avoids the nearly-linear 1-D case). Temperature affects
efficiency: the PEM model shows ~15% higher specific production at 80°C vs 40°C across the
operating range, following Nernst and Arrhenius dependencies.

In [ ]:
# Minimal YAML for a 1 MW single-stack PEM electrolyzer
YAML_TEMPLATE = """
general:
    verbose: False
electrolyzer:
    dt: 1
    supervisor:
        n_stacks: 1
    controller:
        control_type: BaselineDeg
    stack:
        cell_type: PEM
        max_current: 2000
        temperature: {temp_c}
        n_cells: 100
        stack_rating_kW: 1000
"""

TEMPERATURES = [40, 50, 60, 70, 80]  # °C
POWERS_KW = np.linspace(150, 1000, 30)  # kW (above minimum operating power ~130 kW)

print("Generating training data from NatLabRockies electrolyzer simulator...")
X_raw, y_raw = [], []

for T in TEMPERATURES:
    with tempfile.NamedTemporaryFile(suffix=".yaml", mode="w", delete=False) as f:
        f.write(YAML_TEMPLATE.format(temp_c=T))
        yaml_path = f.name

    # Efficient block-constant signal: one 700-step warm-up (PEM turn-on delay ~600 s),
    # then 30 power blocks of 100 steps each; steady-state extracted from last 20 steps.
    signal = np.concatenate(
        [
            np.full(700, POWERS_KW[0] * 1e3),
            *[np.full(100, P * 1e3) for P in POWERS_KW],
        ]
    )
    _, results = run_electrolyzer(yaml_path, signal)
    os.unlink(yaml_path)

    for i, P in enumerate(POWERS_KW):
        start = 700 + i * 100
        h2 = results["kg_rate"].iloc[start + 80 : start + 100].mean() * 3600  # kg/h
        X_raw.append([P, T])
        y_raw.append(h2)

    print(
        f"  T={T}°C | H2 range [{y_raw[-30]:.2f}, {y_raw[-1]:.2f}] kg/h  "
        f"(eff {y_raw[-1] / 1000 * 1000:.1f}%)"
    )

X_raw = np.array(X_raw, dtype=np.float32)
y_raw = np.array(y_raw, dtype=np.float32).reshape(-1, 1)
print(f"\n{len(X_raw)} samples | H2 [{y_raw.min():.3f}, {y_raw.max():.3f}] kg/h")

## Step 2 — Smooth Neural Surrogate

`nn.Sequential` with `nn.Softplus` — required by `gurobi-machinelearning`'s PyTorch converter,
which maps each `Softplus(beta)` neuron to $(1/\beta)\ln(1 + e^{\beta x})$ as a Gurobi
nonlinear constraint. This preserves smooth differentiability and allows the nonlinear
barrier to exploit information gradient unlike ReLU/MILP embeddings solved by branch-and-bound. 

In [ ]:
X_mean = X_raw.mean(0)
X_std = X_raw.std(0) + 1e-9
y_mean = float(y_raw.mean())
y_std = float(y_raw.std()) + 1e-9
X_norm = (X_raw - X_mean) / X_std
y_norm = (y_raw - y_mean) / y_std

torch.manual_seed(42)
surrogate = nn.Sequential(
    nn.Linear(2, 16),
    nn.Softplus(beta=2.0),
    nn.Linear(16, 16),
    nn.Softplus(beta=2.0),
    nn.Linear(16, 1),
)
loader = DataLoader(
    TensorDataset(torch.tensor(X_norm), torch.tensor(y_norm)),
    batch_size=32,
    shuffle=True,
)
opt = torch.optim.Adam(surrogate.parameters(), lr=1e-3)
surrogate.train()
for _ in range(600):
    for xb, yb in loader:
        opt.zero_grad()
        nn.MSELoss()(surrogate(xb), yb).backward()
        opt.step()

surrogate.eval()
with torch.no_grad():
    y_pred = surrogate(torch.tensor(X_norm)).numpy().ravel() * y_std + y_mean
rmse = float(np.sqrt(np.mean((y_pred - y_raw.ravel()) ** 2)))
print(f"Surrogate RMSE: {rmse:.4f} kg/h ({rmse / y_raw.max() * 100:.2f}% of max)")

fig, ax = plt.subplots(figsize=(7, 3.5))
colors = plt.cm.plasma(np.linspace(0.15, 0.85, len(TEMPERATURES)))
for T, col in zip(TEMPERATURES, colors):
    mask = X_raw[:, 1] == T
    P_sorted = np.sort(POWERS_KW)
    X_q = np.column_stack([P_sorted, np.full_like(P_sorted, T)]).astype(np.float32)
    with torch.no_grad():
        H2_nn = (
            surrogate(torch.tensor((X_q - X_mean) / X_std)).numpy().ravel() * y_std
            + y_mean
        )
    H2_sim = y_raw[mask].ravel()
    ax.plot(P_sorted, H2_sim, "o", color=col, ms=4, label=f"{T}°C (sim)")
    ax.plot(P_sorted, H2_nn, "-", color=col, lw=2, label=f"{T}°C (NN)")
ax.set_xlabel("Power (kW)")
ax.set_ylabel("H₂ (kg/h)")
ax.set_title("Surrogate fit: NatLabRockies PEM data vs Softplus NN")
ax.legend(ncol=2, fontsize=7)
plt.tight_layout()
plt.show()

## Step 3 — Dispatch Optimisation with Gurobi

Maximise 24-hour profit: $\sum_t [p_{\text{H}_2} \hat{m}(P_t, T_t) - c_t P_t]$
where $p_{\text{H}_2} = 5$ \$/kg (target green H₂ price) and $c_t$ is the electricity spot
price. Temperature $T_t$ follows a known warm-up profile. Ramp limits reflect real electrolyzer
dynamics ($\Delta P \leq 200$ kW/h). The Softplus surrogate is embedded via `add_predictor_constr`;
normalisation is handled by explicit Gurobi linear constraints.

In [ ]:
T_STEPS = 24
H2_PRICE = 5.0  # $/kg
np.random.seed(7)
price = (
    np.maximum(
        5
        + 4 * np.sin(np.linspace(-np.pi / 2, 3 * np.pi / 2, T_STEPS))
        + np.random.randn(T_STEPS) * 0.5,
        1.0,
    )
    / 100.0
)  # $/kWh, range ~0.01-0.09
T_prof = 40.0 + 40.0 * (1 - np.exp(-np.arange(T_STEPS) / 4.0))  # warm-up 40→80°C

m = gp.Model("dispatch")
m.Params.OutputFlag = 1
m.Params.NonConvex = 2
m.Params.OptimalityTarget = 1

P = m.addMVar((T_STEPS, 1), lb=150.0, ub=1000.0, name="P")  # kW
H = m.addMVar((T_STEPS, 1), lb=0.0, name="H")  # kg/h
PT_n = m.addMVar((T_STEPS, 2), lb=-10.0, ub=10.0, name="PT_n")
H_n = m.addMVar((T_STEPS, 1), lb=-10.0, ub=10.0, name="H_n")

for t in range(T_STEPS):
    m.addConstr(PT_n[t, 0] == (P[t, 0] - X_mean[0]) / X_std[0])
    m.addConstr(PT_n[t, 1] == (T_prof[t] - X_mean[1]) / X_std[1])
    add_predictor_constr(m, surrogate, PT_n[t : t + 1, :], H_n[t : t + 1, :])

m.addConstr(H == H_n * y_std + y_mean)
for t in range(1, T_STEPS):
    m.addConstr(P[t, 0] - P[t - 1, 0] <= 200.0)
    m.addConstr(P[t, 0] - P[t - 1, 0] >= -200.0)

# Profit objective: H2 revenue minus electricity cost
m.setObjective(
    gp.quicksum(H2_PRICE * H[t, 0] - price[t] * P[t, 0] for t in range(T_STEPS)),
    GRB.MAXIMIZE,
)
P.Start = np.full((T_STEPS, 1), 575.0)  # warm start at midpoint

t0 = time.time()
m.optimize()
solve_time = time.time() - t0
status = {GRB.OPTIMAL: "Optimal", GRB.LOCALLY_OPTIMAL: "Locally optimal"}.get(
    m.Status, str(m.Status)
)
print(f"Status: {status}  |  Time: {solve_time:.2f}s", end="")
if m.Status in (GRB.OPTIMAL, GRB.LOCALLY_OPTIMAL):
    print(f"  |  Obj: ${m.ObjVal:.2f}")
else:
    print()

## Results

In [ ]:
if m.Status in (GRB.OPTIMAL, GRB.LOCALLY_OPTIMAL):
    P_opt = P.X.ravel()
    H_opt = H.X.ravel()
    hours = np.arange(T_STEPS)
    breakeven = H2_PRICE * float(
        np.diff(y_raw.ravel()[:30]).mean() / np.diff(POWERS_KW).mean()
    )  # $/kWh at margin

    fig, axes = plt.subplots(3, 1, figsize=(9, 7), sharex=True)

    ax0b = axes[0].twinx()
    axes[0].step(hours, P_opt, where="post", lw=2, color="tab:blue")
    ax0b.plot(hours, price * 100, "r--", alpha=0.7, label="Price (¢/kWh)")
    ax0b.axhline(
        breakeven * 100,
        color="r",
        linestyle=":",
        alpha=0.5,
        label=f"Breakeven ({breakeven * 100:.1f}¢)",
    )
    axes[0].set_ylabel("Power (kW)")
    ax0b.set_ylabel("Price (¢/kWh)", color="r")
    ax0b.legend(fontsize=8)
    axes[0].set_title("Electrolyzer Dispatch — Softplus Surrogate + Gurobi NLP Barrier")

    axes[1].plot(hours, T_prof, "o-", color="tab:red", ms=4, lw=2)
    axes[1].set_ylabel("Stack Temp (°C)")

    axes[2].step(hours, H_opt, where="post", lw=2, color="tab:green")
    axes[2].set_ylabel("H₂ rate (kg/h)")
    axes[2].set_xlabel("Hour")

    plt.tight_layout()
    plt.show()
    print(
        f"Solve: {solve_time:.2f}s  |  Revenue: ${m.ObjVal:.2f}  |  "
        f"Total H₂: {H_opt.sum():.1f} kg  |  Surrogate RMSE: {rmse:.4f} kg/h"
    )

*Copyright © 2026 Gurobi Optimization, LLC — Apache 2.0 License*